# Flink × dbt × Iceberg — PoC Demo

An end-to-end walkthrough of the **two-pipeline architecture** in this repo.

**What this notebook does:**
1. Verifies the Docker stack is running (Kafka, Flink JobManager/TaskManager, SQL Gateway)
2. Talks to the Flink SQL Gateway directly over REST
3. Inspects the streaming + batch SQL scripts side by side
4. Walks through the dbt project: sources → models → tests → docs
5. Demonstrates the CDC current-state and GDPR soft-delete patterns
6. Shows compiled SQL output to prove Jinja → Flink SQL translation

**To run:** start kernel against `.venv39` (or any Python 3.9+ with `requests` installed).

## 1 · Environment & infrastructure check

Before anything else — confirm the cluster is up. The PoC needs:
- **Kafka** + **Zookeeper** (streaming source)
- **Flink JobManager** (port 8081 — Web UI)
- **Flink TaskManager** (executes the SQL jobs)
- **Flink SQL Gateway** (port 8083 — what dbt talks to)

In [ ]:
!docker ps --format 'table {{.Names}}\t{{.Status}}\t{{.Ports}}' | grep -E 'flink|kafka|zookeeper|NAMES'

In [ ]:
import requests, json

GATEWAY = 'http://localhost:8083'
FLINK_UI = 'http://localhost:8081'

info = requests.get(f'{GATEWAY}/v1/info').json()
overview = requests.get(f'{FLINK_UI}/overview').json()

print(f"Flink SQL Gateway version : {info.get('productName', '?')} {info.get('version', '?')}")
print(f"Flink cluster slots       : {overview['slots-available']}/{overview['slots-total']} free")
print(f"Flink TaskManagers        : {overview['taskmanagers']}")
print(f"Running jobs              : {overview['jobs-running']}")
print(f"Finished jobs             : {overview['jobs-finished']}")

## 2 · The two pipelines, side by side

```
  ┌──────────┐    ┌────────┐    ┌──────────────────┐
  │  Kafka   │───▶│ Flink  │───▶│  Iceberg CDC log │ ◀── streaming (long-lived)
  │ (orders) │    │ stream │    │  (append-only)   │
  └──────────┘    └────────┘    └──────────────────┘
                                          │
                                          ▼
                                   ┌──────────────┐
                                   │     dbt      │  → marts, dashboards
                                   └──────────────┘
                                          ▲
  ┌──────────┐    ┌────────┐    ┌──────────────────┐
  │ S3 file  │───▶│ Flink  │───▶│ Iceberg tables   │ ◀── batch (one-shot)
  │ Parquet  │    │ batch  │    │ (final commit)   │
  └──────────┘    └────────┘    └──────────────────┘
```

Both lanes write to the **same Iceberg warehouse** in the **same Glue catalog**.
They typically write to *different tables* and dbt joins them downstream.

In [ ]:
# Inspect the SQL scripts that drive each lane
import os
from pathlib import Path

ROOT = Path('/Users/saikumar/Documents/nbp/Github_repos/flink_dbt_poc')

print('━━━ STREAMING SQL SCRIPTS ━━━')
for p in sorted((ROOT / 'sql').glob('streaming_*.sql')):
    size = p.stat().st_size
    print(f'  {p.name:40s}  {size:>6,} bytes')

print()
print('━━━ BATCH SQL SCRIPTS ━━━')
for p in sorted((ROOT / 'sql').glob('batch_*.sql')):
    size = p.stat().st_size
    print(f'  {p.name:40s}  {size:>6,} bytes')

print()
print('━━━ COMBINED PIPELINES ━━━')
for p in sorted((ROOT / 'sql').glob('flink_*.sql')):
    print(f'  {p.name:40s}  {p.stat().st_size:>6,} bytes')

## 3 · The streaming pipeline — Kafka headers as CDC metadata

The streaming script reads Kafka messages **and their headers** as virtual METADATA columns. The CDC `operation` (INSERT/UPDATE/DELETE) travels in the header so the payload schema stays clean.

In [ ]:
!head -60 /Users/saikumar/Documents/nbp/Github_repos/flink_dbt_poc/sql/streaming_cdc_op_field.sql

## 4 · Talk to the Flink SQL Gateway directly

dbt is just a thin client over this REST API. Let's send a query ourselves to prove the round-trip works.

In [ ]:
import time, requests

BASE = 'http://localhost:8083/v1'

def open_session():
    r = requests.post(f'{BASE}/sessions', json={'sessionName': 'notebook_demo'})
    r.raise_for_status()
    return r.json()['sessionHandle']

def run_sql(session, sql):
    r = requests.post(f'{BASE}/sessions/{session}/statements', json={'statement': sql})
    r.raise_for_status()
    return r.json()['operationHandle']

def fetch(session, op, token=0):
    r = requests.get(f'{BASE}/sessions/{session}/operations/{op}/result/{token}')
    r.raise_for_status()
    return r.json()

def wait_and_collect(session, op, max_pages=10):
    rows, token = [], 0
    for _ in range(max_pages):
        page = fetch(session, op, token)
        rows.extend(page.get('results', {}).get('data', []))
        next_uri = page.get('nextResultUri')
        if not next_uri:
            break
        token = int(next_uri.split('/')[-1])
        time.sleep(0.2)
    return rows, page.get('results', {}).get('columns', [])

session = open_session()
print(f'session opened: {session[:8]}…')

In [ ]:
# Sanity-check: what catalogs does the gateway see?
op = run_sql(session, 'SHOW CATALOGS')
rows, cols = wait_and_collect(session, op)

print('Catalogs visible to Flink:')
for r in rows:
    print(' ', r['fields'])

## 5 · Inspect the dbt project

Everything dbt knows about the project.

In [ ]:
%%bash
cd /Users/saikumar/Documents/nbp/Github_repos/flink_dbt_poc
source .venv39/bin/activate
dbt list --profiles-dir dbt 2>&1 | grep -E '^(flink_dbt_poc|source:)' | head -25

## 6 · The CDC current-state macro — Jinja → real SQL

The macro takes parameters and writes the window-dedup pattern. dbt expands it at compile time.

In [ ]:
print('═══ MACRO DEFINITION ═══')
print(open(ROOT / 'dbt/macros/cdc_current_state.sql').read())

print()
print('═══ MODEL THAT CALLS THE MACRO ═══')
print(open(ROOT / 'dbt/models/mart_orders_cdc_current.sql').read())

In [ ]:
%%bash
cd /Users/saikumar/Documents/nbp/Github_repos/flink_dbt_poc
source .venv39/bin/activate
dbt compile --select mart_orders_cdc_current --profiles-dir dbt 2>&1 | tail -3
echo
echo '═══ COMPILED SQL (sent to Flink) ═══'
cat target/compiled/flink_dbt_poc/dbt/models/mart_orders_cdc_current.sql

## 7 · Run the dbt models against Flink

Materializes 4 view models against the live SQL Gateway. Each becomes a Flink VIEW backed by an Iceberg query.

In [ ]:
%%bash
cd /Users/saikumar/Documents/nbp/Github_repos/flink_dbt_poc
source .venv39/bin/activate
# exclude the artifacts that depend on the not-yet-typed source declarations
dbt run \
  --select stg_orders mart_orders_cdc_current mart_orders_active mart_orders_summary \
  --profiles-dir dbt 2>&1 \
  | grep -aE 'OK created|of [0-9]+ |Finished running|Done\.|Concurrency|Found |Completed'

## 8 · The GDPR soft-delete pattern

Streaming jobs can't safely hard-delete from a row they're actively writing — Iceberg would conflict.  
Pattern: streaming flips `is_deleted=TRUE`; a scheduled batch job purges flagged rows after the legal retention window.

In [ ]:
print('═══ SOFT-DELETE MACRO ═══')
print(open(ROOT / 'dbt/macros/soft_delete_active.sql').read())

print()
print('═══ MODEL ═══')
print(open(ROOT / 'dbt/models/mart_orders_active.sql').read())

In [ ]:
%%bash
cat /Users/saikumar/Documents/nbp/Github_repos/flink_dbt_poc/target/compiled/flink_dbt_poc/dbt/models/mart_orders_active.sql

## 9 · The incremental fact — `is_incremental()` in action

The fact table merges new rows on every run. First run = full build, subsequent runs = only rows newer than `max(order_date)`.

In [ ]:
print(open(ROOT / 'dbt/models/marts/fct_daily_revenue.sql').read())

In [ ]:
%%bash
cd /Users/saikumar/Documents/nbp/Github_repos/flink_dbt_poc
source .venv39/bin/activate
dbt compile --select fct_daily_revenue --profiles-dir dbt 2>&1 | tail -3
echo
echo '═══ COMPILED (first run — incremental guard collapsed) ═══'
cat target/compiled/flink_dbt_poc/dbt/models/marts/fct_daily_revenue.sql

## 10 · The lineage graph

dbt builds a DAG from every `ref()` and `source()` call. View it in the browser:

In [ ]:
import json
from pathlib import Path

manifest = json.loads(Path('/Users/saikumar/Documents/nbp/Github_repos/flink_dbt_poc/target/manifest.json').read_text())

by_type = {}
for node_id, node in manifest['nodes'].items():
    by_type.setdefault(node['resource_type'], []).append(node_id.split('.')[-1])
for node_id, src in manifest['sources'].items():
    by_type.setdefault('source', []).append(f"{src['source_name']}.{src['name']}")
for node_id, exp in manifest.get('exposures', {}).items():
    by_type.setdefault('exposure', []).append(exp['name'])

print('━━━ dbt project inventory ━━━')
for t, names in sorted(by_type.items()):
    print(f'  {t:12s}  {len(names):3d}')

print()
print('━━━ DAG edges (model → depends on) ━━━')
for nid, node in manifest['nodes'].items():
    if node['resource_type'] != 'model': continue
    deps = node.get('depends_on', {}).get('nodes', [])
    pretty = [d.split('.')[-1] for d in deps if not d.startswith('macro')]
    if pretty:
        print(f"  {node['name']:30s} ← {', '.join(pretty)}")

In [ ]:
# To open the interactive lineage graph in your browser:
import subprocess, webbrowser, time

ROOT = '/Users/saikumar/Documents/nbp/Github_repos/flink_dbt_poc'
print('To start the dbt docs server, run in a terminal:')
print(f'  cd {ROOT} && source .venv39/bin/activate && dbt docs serve --profiles-dir dbt')
print()
print('Then open: http://localhost:8080')
print()
print('Or click below to open it now (assumes server is already running):')
# Uncomment to auto-open:
# webbrowser.open('http://localhost:8080')

## 11 · Cleanup

In [ ]:
# close the SQL Gateway session we opened earlier
try:
    requests.delete(f'{BASE}/sessions/{session}')
    print(f'session {session[:8]}… closed')
except Exception as e:
    print(f'cleanup skipped: {e}')

## Summary — what this notebook proved

| Capability | Where to look |
|---|---|
| Live Flink + Kafka + SQL Gateway | Cell 1 (docker ps) + Cell 2 (REST `/info`) |
| Direct REST control of Flink | Cell 6 (`SHOW CATALOGS`) |
| Reusable SQL via Jinja macros | Cell 8 (`cdc_current_state` definition + compile output) |
| dbt models materialized against Flink | Cell 9 (4 of 4 views FINISHED) |
| GDPR-safe soft-delete pattern | Cell 10 (`soft_delete_active`) |
| Incremental fact with `is_incremental()` | Cell 11 |
| Lineage graph (DAG) | Cell 12 (parsed manifest) + Cell 13 (docs server) |

### Key observations
- The same SQL surface (dbt) targets both streaming and batch modes
- Macros eliminate copy-paste of CDC dedup logic across models
- Iceberg + Glue means everything dbt builds is portable to Athena, Trino, Spark
- dbt-flink adapter is young — `seed`, `snapshot`, source auto-registration have rough edges
- For a production migration off Spark/Glue: adopt **dbt on top of existing Spark first**, evaluate Flink only for sub-10s latency or stateful-windowing use cases
